In [1]:
import gc
import time
import warnings
import numpy as np
import pandas as pd

from os.path import join
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [2]:
folder = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU'

train_df = pd.read_csv(join(folder, 'train_df.csv'), index_col=[0, 1], parse_dates=[1])
train_target = pd.read_csv(join(folder, 'train_target.csv'), index_col=[0, 1], parse_dates=[1])

val_df = pd.read_csv(join(folder, 'val_df.csv'), index_col=[0, 1], parse_dates=[1])
val_target = pd.read_csv(join(folder, 'val_target.csv'), index_col=[0, 1], parse_dates=[1])

test_df = pd.read_csv(join(folder, 'test_df.csv'), index_col=[0, 1], parse_dates=[1])
test_target = pd.read_csv(join(folder, 'test_target.csv'), index_col=[0, 1], parse_dates=[1])

In [3]:
def reduce_memory_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].astype(np.float32)
        elif pd.api.types.is_integer_dtype(out[col]):
            cmin, cmax = out[col].min(), out[col].max()

            if cmin >= np.iinfo(np.int8).min and cmax <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif cmin >= np.iinfo(np.int16).min and cmax <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

    return out


def prepare_features(df: pd.DataFrame) -> np.ndarray:
    X = df.copy()

    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = pd.to_numeric(X[col], errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    X = X.fillna(0)
    X = reduce_memory_df(X)

    return X.to_numpy(dtype=np.float32, copy=False)


def extract_target(target_obj):
    if isinstance(target_obj, pd.DataFrame):
        if "fault_type" in target_obj.columns:
            return target_obj["fault_type"].values
        return target_obj.iloc[:, 0].values
    return np.asarray(target_obj)

In [4]:
X_train_np = prepare_features(train_df)
X_val_np = prepare_features(val_df)
X_test_np = prepare_features(test_df)

y_train_raw = extract_target(train_target)
y_val_raw = extract_target(val_target)
y_test_raw = extract_target(test_target)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)

del train_df, val_df, test_df
del train_target, val_target, test_target
del y_train_raw, y_val_raw, y_test_raw
gc.collect()

print("X_train:", X_train_np.shape, X_train_np.dtype)
print("X_val  :", X_val_np.shape, X_val_np.dtype)
print("X_test :", X_test_np.shape, X_test_np.dtype)
print("Classes:", len(label_encoder.classes_))

X_train: (4448700, 25) float32
X_val  : (1317600, 25) float32
X_test : (1899361, 25) float32
Classes: 15


In [5]:
def stratified_subsample_np(X, y, n_samples, random_state=42):
    if n_samples >= len(X):
        return X, y

    rng = np.random.RandomState(random_state)
    y = np.asarray(y)

    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / counts.sum()

    target_counts = np.maximum(1, np.floor(proportions * n_samples).astype(int))
    diff = n_samples - target_counts.sum()

    if diff > 0:
        order = np.argsort(-proportions)
        for i in range(diff):
            target_counts[order[i % len(order)]] += 1
    elif diff < 0:
        order = np.argsort(target_counts)[::-1]
        to_remove = -diff
        j = 0
        while to_remove > 0 and j < len(order) * 10:
            idx = order[j % len(order)]
            if target_counts[idx] > 1:
                target_counts[idx] -= 1
                to_remove -= 1
            j += 1

    selected_idx = []
    for cls, need in zip(classes, target_counts):
        cls_idx = np.where(y == cls)[0]
        if len(cls_idx) <= need:
            take = cls_idx
        else:
            take = rng.choice(cls_idx, size=need, replace=False)
        selected_idx.append(take)

    selected_idx = np.concatenate(selected_idx)
    rng.shuffle(selected_idx)

    return X[selected_idx], y[selected_idx]

In [6]:
RANDOM_STATE = 42
N_JOBS = 2

baseline_params = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "class_weight": None,
}

configs = [
    {**baseline_params, "sample_size": 400_000},
    {**baseline_params, "sample_size": 450_000},
    {
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": None,
        "sample_size": 400_000,
    },
    {
        "n_estimators": 300,
        "max_depth": 25,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": None,
        "sample_size": 400_000,
    },
]

baseline_config = {**baseline_params, "sample_size": 400_000}

print("Total configs:", len(configs))
configs

Total configs: 4


[{'n_estimators': 300,
  'max_depth': None,
  'min_samples_split': 2,
  'min_samples_leaf': 2,
  'max_features': 'sqrt',
  'class_weight': None,
  'sample_size': 400000},
 {'n_estimators': 300,
  'max_depth': None,
  'min_samples_split': 2,
  'min_samples_leaf': 2,
  'max_features': 'sqrt',
  'class_weight': None,
  'sample_size': 450000},
 {'n_estimators': 500,
  'max_depth': None,
  'min_samples_split': 2,
  'min_samples_leaf': 2,
  'max_features': 'sqrt',
  'class_weight': None,
  'sample_size': 400000},
 {'n_estimators': 300,
  'max_depth': 25,
  'min_samples_split': 2,
  'min_samples_leaf': 1,
  'max_features': 'sqrt',
  'class_weight': None,
  'sample_size': 400000}]

In [7]:
def evaluate_config(config):
    sample_size = config["sample_size"]
    model_params = {k: v for k, v in config.items() if k != "sample_size"}

    X_sub, y_sub = stratified_subsample_np(
        X_train_np,
        y_train,
        n_samples=sample_size,
        random_state=RANDOM_STATE
    )

    model = ExtraTreesClassifier(
        **model_params,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbose=0,
    )

    t0 = time.time()
    model.fit(X_sub, y_sub)
    fit_seconds = time.time() - t0

    val_pred = model.predict(X_val_np)
    val_proba = model.predict_proba(X_val_np)

    metrics = {
        **config,
        "is_baseline": config == baseline_config,
        "train_rows": len(X_sub),
        "fit_seconds": round(fit_seconds, 3),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_macro_f1": f1_score(y_val, val_pred, average="macro"),
        "val_weighted_f1": f1_score(y_val, val_pred, average="weighted"),
        "val_logloss": log_loss(
            y_val,
            val_proba,
            labels=np.arange(len(label_encoder.classes_))
        ),
    }

    del X_sub, y_sub, val_pred, val_proba
    gc.collect()

    return metrics, model

In [8]:
results = []
best_model = None
best_metrics = None
best_score = None

for i, config in enumerate(configs, start=1):
    print(f"[{i}/{len(configs)}] {config}")

    metrics, model = evaluate_config(config)
    results.append(metrics)

    score = (
        metrics["val_macro_f1"],
        metrics["val_weighted_f1"],
        metrics["val_accuracy"]
    )

    if best_score is None or score > best_score:
        if best_model is not None:
            del best_model
            gc.collect()

        best_model = model
        best_metrics = metrics
        best_score = score
        print("  -> new best")
    else:
        del model
        gc.collect()

[1/4] {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': None, 'sample_size': 400000}
  -> new best
[2/4] {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': None, 'sample_size': 450000}
  -> new best
[3/4] {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': None, 'sample_size': 400000}
  -> new best
[4/4] {'n_estimators': 300, 'max_depth': 25, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': None, 'sample_size': 400000}


In [9]:
results_df = pd.DataFrame(results).sort_values(
    ["val_macro_f1", "val_weighted_f1", "val_accuracy"],
    ascending=False
).reset_index(drop=True)

results_df

,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,class_weight,sample_size,is_baseline,train_rows,fit_seconds,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss
0,500,NaN,2,2,sqrt,None,400000,False,400000,150.637,0.866124,0.865588,0.865588,0.373410
1,300,NaN,2,2,sqrt,None,450000,False,450000,110.109,0.866044,0.865488,0.865488,0.372349
2,300,NaN,2,2,sqrt,None,400000,True,400000,88.424,0.865409,0.864880,0.864880,0.374045
3,300,25.0,2,1,sqrt,None,400000,False,400000,93.057,0.855168,0.854592,0.854592,0.415226


In [10]:
best_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight=None,
    n_jobs=2,
    random_state=42,
    verbose=0,
)

X_sub, y_sub = stratified_subsample_np(
    X_train_np,
    y_train,
    n_samples=400_000,
    random_state=42
)

best_model.fit(X_sub, y_sub)

test_pred = best_model.predict(X_test_np)
test_proba = best_model.predict_proba(X_test_np)

print("TEST METRICS")
print("accuracy     :", accuracy_score(y_test, test_pred))
print("macro_f1     :", f1_score(y_test, test_pred, average="macro"))
print("weighted_f1  :", f1_score(y_test, test_pred, average="weighted"))
print("logloss      :", log_loss(
    y_test,
    test_proba,
    labels=np.arange(len(label_encoder.classes_))
))

TEST METRICS
accuracy     : 0.7960408790114148
macro_f1     : 0.8027679561672251
weighted_f1  : 0.7956014114013815
logloss      : 0.5294478071650515
